# Blackboard DEMO

## Setup

In [1]:
from src.board import Board

question = """
Целеполагание г. Гатчина (Ленинградская область) на 2026-2035 годы
"""

board = Board(question=question)

In [2]:
from src.agents.factories import *

generator_agent = create_generator_agent(board)
roles = generator_agent.invoke([]).roles

In [3]:
expert_agents = [create_expert_agent(r, board) for r in roles]

for expert in expert_agents:
    print(str.join(', ', expert.info.values()))

27f1b8, Градостроительный планировщик, Эксперт в разработке долгосрочных планов развития городской инфраструктуры и земельного использования.
e4c3bf, Экономист регионального развития, Анализирует экономический потенциал, инвестиционные проекты и финансовую устойчивость Гатчины.
0d2676, Стратег по публичной политике и управлению, Разрабатывает цели и показатели, координирует взаимодействие органов власти и общественности.


In [4]:
from src import get_storage_dir, iter_document_paths
from src.trace_log import get_trace_log_path

storage_dir = get_storage_dir()
document_paths = iter_document_paths(storage_dir)
trace_log_path = get_trace_log_path()

web_expert = create_web_research_expert(board)
document_experts = create_document_research_experts(board, storage_dir=storage_dir)

planner_agent = create_planner_agent(board)
critic_agent = create_critic_agent(board)
cleaner_agent = create_cleaner_agent(board)
decider_agent = create_decider_agent(board)

role_agents = [planner_agent, *expert_agents, web_expert, *document_experts, critic_agent, cleaner_agent, decider_agent]
controller_agent = create_controller_agent(role_agents, board)

print(f"storage: {storage_dir}")
print(f"trace log: {trace_log_path}")
if document_paths:
    print("Найдены документы:")
    for document_path in document_paths:
        print(f"- {document_path.name}")
else:
    print("Документы не найдены в storage; document-эксперты пропущены.")


storage: D:\programming\github\nirma\storage
trace log: D:\programming\github\nirma\logs\trace-20260325T145251Z-5bfbed0c.jsonl
Найдены документы:
- ССЭР Гатчинский.docx
- СТП Гатчинский.pdf


In [5]:
for agent in [*role_agents, controller_agent]:
    response_format = getattr(agent, 'response_format', None)
    checkpointer = getattr(agent, 'checkpointer', None)
    if response_format is not None and checkpointer is not None:
        checkpointer.allowed_msgpack_modules = [
            (response_format.__module__, response_format.__name__),
        ]


## Experiment

In [6]:
def _get_agent(agent_id: str, agents: list):
    for agent in agents:
        if agent.id == agent_id:
            return agent
    return None


def get_agents(agents_ids: list[str], agents: list):
    result = []
    for agent_id in agents_ids:
        agent = _get_agent(agent_id, agents)
        if agent is not None:
            result.append(agent)
    return result


In [7]:
import traceback
from tqdm import tqdm
from langchain.messages import AIMessage, HumanMessage, SystemMessage

is_final = False

for i in range(3):
    print(f'Итерация {i+1}')
    agents_ids = controller_agent.invoke(
        [SystemMessage(f'Записи на доске:\n{str(board.notes)}')], force=True
    ).agents_ids
    agents = get_agents(agents_ids, role_agents)
    print(str.join(', ', [a.role.name for a in agents]))

    for a in tqdm(agents):
        try:
            response = a.invoke([SystemMessage(f'Записи на доске:\n{str(board.notes)}')], force=False)
        except Exception as e:
            print(f'{a.role.name} перегрелся: {type(e).__name__}')
            traceback.print_exc()
            continue

        if a == decider_agent: # DECIDER AGENT
            note = response.note
            is_final = response.is_final
        elif a == cleaner_agent: # CLEANER AGENT
            notes_ids = response.notes_ids
            for note_id in notes_ids:
                board.remove_note(note_id)
            note = response.note
            continue
        else: # OTHER AGENTS
            note = response
        
        note_id = board.add_note(note, a.id, a.role.name)

        if is_final:
            break

    if is_final:
        break

Итерация 1
Веб-эксперт, Документ-эксперт: ССЭР Гатчинский, Документ-эксперт: СТП Гатчинский, Планировщик, Градостроительный планировщик, Экономист регионального развития, Стратег по публичной политике и управлению, Критик, Уборщик, Арбитр


 90%|█████████ | 9/10 [03:52<00:25, 25.80s/it]


In [8]:
board.notes

[Note(content='Документ ССЭР Гатчинский выявляет ключевые проблемы и возможности, которые следует учесть при формулировании целей Гатчины на 2026‑2035 гг.:\n- Состояние коммунальной инфраструктуры критическое: износ водоснабжения и водоотведения > 80 %, очистных сооружений > 85 %; износ теплоснабжения > 60 %, тепловых сетей > 75 %; количество аварий в год на 1000 км сетей составляет 197 (тепло), 253 (водоснабжение) и 150 (водоотведение) [1].\n- Доля сточных вод, очищенных до нормативов, < 40 % от общего объёма, что ограничивает экологическую безопасность [1].\n- Газификация к 01.07.2018 г. составляет 69 %; 172 из 240 населённых пунктов района остаются негазифицированными, что требует значительных инвестиций в магистральные и межпоселковые газопроводы [1].\n- Недостаток инвестиций отмечен как основной фактор задержки модернизации коммунальных систем [1].\n- По стратегии социально‑экономического развития Ленобласти до 2030 г., Гатчина и Гатчинский район позиционируются как один из ведущи

In [9]:
board.print()

╭───────────────────────── 📌 Документ-эксперт: ССЭР Гатчинский [4d6c39] ──────────────────────────╮
│ Документ ССЭР Гатчинский выявляет ключевые проблемы и возможности, которые следует учесть при    │
│ формулировании целей Гатчины на 2026‑2035 гг.:                                                   │
│                                                                                                  │
│  • Состояние коммунальной инфраструктуры критическое: износ водоснабжения и водоотведения > 80   │
│    %, очистных сооружений > 85 %; износ тепло...                                                 │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ Документ ССЭР Гатчинский выявляет ключевые проблемы и возможности, которые следует учесть при    │
│ формулировании целей Гатчины на 2026‑2035 гг.:                                                   │
│                                                                                                  │
│  • Состояние коммунальной инфраструктуры критическое: износ водоснабжения и водоотведения > 80   │
│    %, очистных сооружений > 85 %; износ теплоснабжения > 60 %, тепловых сетей > 75 %; количество │
│    аварий в год на 1000 км сетей составляет 197 (тепло), 253 (водоснабжение) и 150               │
│    (водоотведение) [1].                                                                          │
│  • Доля сточных вод, очищенных до нормативов, < 40 % от общего объёма, что ограничивает          │
│    экологическую безопасность [1].                                                               │
│  • Газификация к 01.07.2018 г. составляет 69 %; 172 из 240 населённых пунктов района остаются    │
│    негазифицированными, что требует значительных инвестиций в магистральные и межпоселковые      │
│    газопроводы [1].                                                                              │
│  • Недостаток инвестиций отмечен как основной фактор задержки модернизации коммунальных систем   │
│    [1].                                                                                          │
│  • По стратегии социально‑экономического развития Ленобласти до 2030 г., Гатчина и Гатчинский    │
│    район позиционируются как один из ведущих центров формирования инновационной экономики, с     │
│    приоритетом развития в рамках агломерации Санкт‑Петербурга. Ключевые ресурсы:                 │
│    культурно‑историческое наследие (Гатчинский и Приоратский дворцы), инновационная              │
│    инфраструктура (Петербургский институт ядерной физики, планируемый технопарк «Роснано», вузы) │
│    [2].                                                                                          │
│  • В документе описаны программы 2018‑2020 гг. по развитию культуры, спорта, жилья, социальной   │
│    поддержки и устойчивому общественному развитию, что показывает существующую институциональную │
│    базу для реализации будущих стратегических инициатив (например, поддержка жилья, улучшение    │
│    условий для молодёжи, развитие муниципальных информационных систем) [3][4]. Эти данные        │
│    позволяют сформулировать приоритеты на 2026‑2035 гг.: модернизация и снижение износа          │
│    коммунальных сетей, повышение доли очистки сточных вод до ≥ 80 %, достижение газификации ≥ 90 │
│    %, привлечение инвестиций в инфраструктуру, развитие инновационного технопарка и              │
│    использование культурного наследия как драйвера экономического роста, а также укреп...        │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ keywords: document, research, ссэр, гатчинский                                                   │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── 📌 Документ-эксперт: СТП Гатчинский [80958f] ──────────────────────────╮
│ { "status": "success", "summary": "Документ СТП Гатчинский определяет ключевые количественные    │
│ цели, которые могут стать базой для целеполагания Гатчины на 2026‑2035 гг.: \n- Пропускная       │
│ способность спортивных сооружений: 25‑30 тыс. человек (2020) → 47,5 тыс. (2030)【locator":"...   │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ { "status": "success", "summary": "Документ СТП Гатчинский определяет ключевые количественные    │
│ цели, которые могут стать базой для целеполагания Гатчины на 2026‑2035 гг.: \n- Пропускная       │
│ способность спортивных сооружений: 25‑30 тыс. человек (2020) → 47,5 тыс. (2030)【locator":"СТП   │
│ Гатчинский.pdf#page=20&chunk=55","snippet":"рост единовременной пропускной способности           │
│ спортсооружений ... до 25-30 тыс. человек в 2020 г. и до 47,5 тыс. человек в 2030                │
│ г."},{"locator":"СТП Гатчинский.pdf#page=20&chunk=55","snippet":"увеличение площади спортивных   │
│ залов до 50-55 тыс.м2 ... до 87,5 тыс. м2 в 2030 г."},{"locator":"СТП                            │
│ Гатчинский.pdf#page=21&chunk=56","snippet":"увеличение площади плоскостных сооружений до 350     │
│ тыс. м2 в 2020 г. и до 487,5 тыс. м2 в 2030 г."},{"locator":"СТП                                 │
│ Гатчинский.pdf#page=21&chunk=56","snippet":"наличие к 2030 г. плавательных бассейнов во всех     │
│ городских и сельских поселениях... Статус: предварительный вывод, требуется дополнительная       │
│ проверка.                                                                                        │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ keywords: document, research, стп, гатчинский                                                    │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────── 📌 Планировщик [744ee6] ─────────────────────────────────────╮
│ План действий по сбору данных, анализу, формулированию приоритетов и количественных целей,       │
│ разработке дорожной карты и согласованию стратегии развития Гатчины на 2026‑2035 годы.           │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ План решения задачи целеполагания г. Гатчина на 2026‑2035 годы:                                  │
│                                                                                                  │
│  1 Сбор и систематизация уже имеющихся источников (ССЭР Гатчинский, СТП Гатчинский, региональная │
│    стратегия Ленобласти до 2030 г., муниципальные программы 2018‑2020).                          │
│  2 Выявление пробелов в данных: отсутствие актуальных экономических, демографических,            │
│    транспортных и экологических показателей; запрос недостающей информации у администрации       │
│    города, Росстата, профильных ведомств.                                                        │
│  3 Анализ текущего состояния инфраструктуры и социально‑экономических индикаторов (износ         │
│    коммунальных сетей, доля очистки сточных вод, уровень газификации, потенциал культурного      │
│    наследия, инновационные объекты, спорт‑инфраструктура).                                       │
│  4 Формулирование приоритетных направлений развития: модернизация коммунальной инфраструктуры,   │
│    повышение уровня очистки сточных вод, достижение газификации ≥ 90 %, развитие технопарка и    │
│    инновационной экосистемы, туризм и культурное наследие, спорт и досуг, жильё и                │
│    демографическая политика.                                                                     │
│  5 Определение количественных целей (пример): износ коммунальных сетей < 30 % к 2035 г., очистка │
│    сточных вод ≥ 80 % к 2030 г., газификация ≥ 90 % к 2035 г., пропускная способность            │
│    спортсооружений 47,5 тыс. чел., площадь спортзалов 87,5 тыс м² к 2030 г., наличие             │
│    плавательных бассейнов во всех населённых пунктах к 2035 г.).                                 │
│  6 Разработка дорожной карты с этапами 2026‑2028, 2029‑2032, 2033‑2035, распределение            │
│    инвестиций, определение источников финансирования (бюджет, гранты, частные инвестиции).       │
│  7 Подготовка черновика стратегического документа, согласование с заинтересованными сторонами    │
│    (муниципальные органы, бизнес, научные учреждения, общественные организации, население).      │
│  8 Итоговое оформление рекомендаций и планов реализации, подготовка системы мониторинга и оценки │
│    выполнения целей.                                                                             │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ keywords: стратегия, цели, Гатчина, инфраструктура, анализ                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────── 📌 Градостроительный планировщик [27f1b8] ────────────────────────────╮
│ Предлагаю структуру целеполагания Гатчины на 2026‑2035 годы: видение, приоритеты, количественные │
│ цели, этапы, управление и финансирование.                                                        │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ На основе уже собранных материалов (ССЭР Гатчинский, СТП Гатчинский, региональная стратегия      │
│ Ленобласти до 2030 г., муниципальные программы 2018‑2020) предлагаю следующую структуру          │
│ целеполагания Гатчины на 2026‑2035 годы:                                                         │
│                                                                                                  │
│ 1️⃣ Видение – Гатчина как интегрированный агломерационный центр Санкт‑Петербурга с высоким        │
│ уровнем качества жизни, устойчивой инфраструктурой и инновационной экономикой.                   │
│                                                                                                  │
│ 2️⃣ Стратегические приоритеты: • Модернизация коммунальной инфраструктуры (сократить износ водо‑  │
│ и теплосетей <30 % к 2035 г.; увеличить долю очистки сточных вод ≥80 % к 2030 г.). • Достижение  │
│ газификации ≥90 % к 2035 г. и развитие альтернативных энергоисточников (солярные, биогазовые     │
│ проекты). • Развитие инновационного кластера (технопарк «Роснано», сотрудничество с ПИЯФ,        │
│ поддержка стартап‑инкубаторов). • Транспортная интеграция: расширение пригородных                │
│ железнодорожных и автобусных линий, создание кольцевой велодорожной сети, улучшение доступа к    │
│ МКАД‑КАД. • Туризм и культурное наследие: реставрация Гатчинского и Приоратского дворцов,        │
│ развитие культурных маршрутов, поддержка событийного туризма. • Спорт и досуг: выполнить         │
│ количественные цели СТП (пропускная способность спортсооружений 47,5 тыс. чел., площадь          │
│ спортзалов 87,5 тыс м² к 2030 г., бассейны во всех населённых пунктах к 2035 г.). • Жильё и      │
│ демография: увеличить долю доступного жилья до 30 % от общего фонда, стимулировать приток        │
│ молодых семей через субсидии и инфраструктурные проекты.                                         │
│                                                                                                  │
│ 3️⃣ Количественные цели (ключевые индикаторы) – см. пункт 2, дополнительно: снижение уровня       │
│ загрязнения воздуха ПМ2.5 до <12 µg/m³ к 2035 г.; увеличение площади зелёных насаждений до 15 %  │
│ территории города.                                                                               │
│                                                                                                  │
│ 4️⃣ Этапы реализации: • 2026‑2028 – аудит инфраструктуры, привлечение инвестиций, запуск пилотных │
│ проектов газификации и энергоэффективности. • 2029‑2032 – строительство и модернизация           │
│ коммунальных сетей, развитие транспортных коридоров, открытие технопарка. • 2033‑2035 –          │
│ завершение целей п...                                                                            │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ keywords: Гатчина, стратегия, инфраструктура, инновации, транспорт                               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📌 Экономист регионального развития  ──────────────────────────────╮
│ Экономический анализ Гатчины с приоритетами в туризме, инновациях, логистике и агропроме,        │
│ финансовыми рекомендациями и индикаторами для 2026‑2035 гг.                                      │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ Экономический анализ потенциала г. Гатчина (Ленобласть) для целеполагания 2026‑2035 гг.          │
│                                                                                                  │
│ 1️⃣ Сектора с наибольшим ростовым потенциалом: • Туризм и культурное наследие – 2‑3 млн руб       │
│ ежегодных доходов от посещения Гатчинского и Приоратского дворцов, развитие событийного туризма  │
│ (фестивали, конференции). • Инновационный кластер – технопарк «Роснано», ПИЯФ,                   │
│ стартап‑инкубаторы; прогнозируемый прирост ВРП в секторе НИОКР ≈ 5 % в год при привлечении       │
│ инвестиций ≈ 2 млрд руб к 2030 г. • Логистика и транспорт – близость к Санкт‑Петербургу,         │
│ развитие железнодорожных и автодорожных коридоров; потенциал создания логистического хаба,       │
│ увеличение товарооборота на ≈ 10 % к 2035 г. • Агропродовольственный комплекс – модернизация     │
│ фермерских хозяйств, развитие фермерских рынков, экспорт продукции в СПб и за рубеж.             │
│                                                                                                  │
│ 2️⃣ Финансовая устойчивость: • Бюджетные ресурсы – планировать рост собственных доходов           │
│ муниципального бюджета за счёт увеличения налоговой базы (НДФЛ, налог на прибыль предприятий) на │
│ ≈ 4 % в год, оптимизировать расходы через цифровизацию управления. • Инвестиционные инструменты  │
│ – ППП (PPP) для модернизации коммунальных сетей, облигации муниципального уровня, гранты Фонда   │
│ «Развитие регионов», привлечение частных инвестиций в технопарк и туристическую инфраструктуру.  │
│ • Экономический эффект – ожидаемое увеличение ВРП города на ≈ 30 % к 2035 г. при реализации      │
│ приоритетных проектов, снижение безработицы до ≤ 4 %.                                            │
│                                                                                                  │
│ 3️⃣ Ключевые экономические индикаторы для мониторинга: • Доля ВРП, генерируемая инновационным     │
│ сектором – цель ≥ 15 % к 2035 г. • Уровень привлечённых инвестиций в инфраструктуру – цель ≥ 5   │
│ млрд руб к 2030 г. • Коэффициент финансовой автономии бюджета – цель ≥ 0,45 к 2035 г. •          │
│ Показатели занятости в приоритетных секторах (туризм, IT, логистика) – рост на ≥ 2 % ежегодно.   │
│                                                                                                  │
│ 4️⃣ Рекомендации: • Создать экономический совет при мэрии, включающий представителей бизнеса,     │
│ академии и инвесторов....                                                                        │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ keywords: экономика, инвестиции, инновации, туризм, финансы                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────── 📌 Стратег по публичной политике и управлению [0d2676] ─────────────────────╮
│ Предлагаю структуру управления, публичного планирования, KPI‑панели, финансовой прозрачности и   │
│ управления рисками для реализации стратегии Гатчины 2026‑2035.                                   │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ Предлагаю добавить к уже сформированным приоритетам и количественным целям Гатчины (2026‑2035)   │
│ блок «Управление и мониторинг стратегии»:                                                        │
│                                                                                                  │
│ 1️⃣ Институциональная платформа – создать Стратегический совет города, включающий мэра,           │
│ заместителей, представителей областного правительства, бизнес‑сообщества, научных учреждений     │
│ (ПИЯФ, технопарк), общественных организаций и граждан (через онлайн‑форум). Совет будет          │
│ координировать реализацию приоритетов, согласовывать бюджеты и оценивать риски.                  │
│                                                                                                  │
│ 2️⃣ Публичный процесс планирования – проводить ежегодные открытые слушания и онлайн‑консультации  │
│ по каждому этапу дорожной карты; формировать «Карту интересов» стейкхолдеров, фиксировать их     │
│ требования и предложения в публичном реестре.                                                    │
│                                                                                                  │
│ 3️⃣ Система KPI и цифровой мониторинг – разработать единую панель индикаторов (износ коммунальных │
│ сетей, доля очистки сточных вод, уровень газификации, ВРП инновационного сектора, количество     │
│ туристических посещений, спортивные пропускные способности, экологические показатели). Данные    │
│ обновлять ежеквартально через муниципальную информационную систему и публиковать в открытом      │
│ доступе.                                                                                         │
│                                                                                                  │
│ 4️⃣ Механизмы корректировки – предусмотреть ежегодный «периодический обзор» стратегии, где на     │
│ основе анализа KPI совет принимает решения о перераспределении ресурсов, корректировке сроков и  │
│ целей.                                                                                           │
│                                                                                                  │
│ 5️⃣ Финансовая прозрачность – публиковать детализированный бюджет реализации проектов, источники  │
│ финансирования (бюджет, гранты, ППП, облигации) и отчёты о расходах. Внедрить независимый аудит  │
│ каждые два года.                                                                                 │
│                                                                                                  │
│ 6️⃣ Управление рисками – создать реестр рисков (финансовых, экологических,                        │
│ социально‑политических) с оценкой вероятности и воздействия; разработать план реагирования и     │
│ резервные фонды.                                                                                 │
│                                                                                                  │
│ Эти элементы обеспечат согласованность действий всех участников, повысит доверие населения и     │
│ инвесторов, а также позволит своевременно корректировать курс стратегии в динамичной среде.      │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ keywords: управление, стратегия, КPI, прозрачность, гражданское участие                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 📌 Арбитр  ───────────────────────────────────────────╮
│ Достаточно информации для завершения задачи по целеполаганию Гатчины на 2026‑2035 годы.          │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ На доске присутствуют ключевые документы и материалы: ССЭР Гатчинский, СТП Гатчинский,           │
│ региональная стратегия Ленобласти до 2030 г., муниципальные программы 2018‑2020, план действий   │
│ по сбору и анализу данных, предложенная структура целеполагания (видение, стратегические         │
│ приоритеты, количественные цели, этапы реализации), экономический анализ с финансовыми           │
│ рекомендациями и индикаторами, а также предложения по управлению, публичному планированию и      │
│ мониторингу стратегии. Эти сведения охватывают все необходимые компоненты целеполагания Гатчины  │
│ на 2026‑2035 годы, включая инфраструктурные, экономические, социальные и экологические аспекты,  │
│ а также механизмы реализации и контроля. Следовательно, команда обладает достаточной информацией │
│ для формирования окончательного ответа.                                                          │
│ ──────────────────────────────────────────────────────────────────────────────────────────────── │
│ keywords: цель, Гатчина, стратегия, информация, достаточно                                       │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

### Summarize

In [10]:
from src.agents import Agent

In [11]:
summarizer_actor_agent = Agent(
    tools=board.tools,  # оставляем инструменты
    system_prompt="""
    Ваша задача: синтезировать финальный ответ на поставленную задачу, основываясь ТОЛЬКО на материалах доски.
    
    Поставленная задача:
    `{question}`
    
    Алгоритм:
    1. Вызовите get_notes(), чтобы увидеть все записи
    2. Вызовите get_note() для КАЖДОЙ из них
    3. Синтезируйте финальный ответ
    4. Верните ТОЛЬКО финальный ответ, без лишних комментариев
    """.format(question=board.question)
)

summarizer_critic_agent = Agent(
    tools=board.tools,  # оставляем инструменты
    system_prompt="""
    Ваша задача: на основе черновика финального ответа на поставленную задачу и материалов на доске выдайте список замечаний.
    
    Поставленная задача:
    `{question}`
    """.format(question=board.question)
)

In [13]:
messages = []

n_gen = 3

for i in tqdm(range(n_gen)):
    actor_res = summarizer_actor_agent.invoke([HumanMessage(m) for m in messages[-1:]])
    messages.append(actor_res)

    if (i+1) < n_gen:
        critic_res = summarizer_critic_agent.invoke([HumanMessage(m) for m in messages[-1:]])
        messages.append(critic_res)

  0%|          | 0/3 [09:19<?, ?it/s]


RuntimeError: {"error":{"message":"error parsing tool call: raw='{\"note Arb  ?  ...\"}', err=invalid character '}' after object key","type":"api_error","param":null,"code":null}}

In [ ]:
board.notes

[Note(content='План решения задачи:\n\n1. **Формулировка задачи** – собрать и систематизировать информацию о текущем состоянии города Гатчина, определить ключевые направления развития и сформировать цели на 2026‑2035 годы.\n\n2. **Сбор исходных данных**\n   - демографические, экономические, социальные и инфраструктурные показатели;\n   - существующие стратегии и программы развития (на уровне муниципалитета, региона, федеральные проекты);\n   - результаты предыдущих планов (2016‑2025) и их оценка.\n\n3. **Анализ заинтересованных сторон**\n   - органы местного самоуправления, бизнес‑сообщество, общественные организации, жители.\n   - проведение интервью/опросов, формирование требований и ожиданий.\n\n4. **SWOT‑анализ** города (сильные/слабые стороны, возможности, угрозы).\n\n5. **Определение стратегических направлений** (например, экономическое развитие, жилищно‑коммунальное хозяйство, транспорт, экология, культура, цифровизация).\n\n6. **Формулирование целей и задач** для каждого направ

In [ ]:
messages

['**Целеполагание г.\u202fГатчина (Ленинградская область)\u202fна 2026‑2035\u202fгоды**  \n\n---\n\n## 1.\u202fОбщая методология (по\u202fплану\u202ff564a3)\n\n1. **Формулировка задачи** – определить текущее состояние города и сформулировать цели развития до\u202f2035\u202fгода.  \n2. **Сбор и систематизация исходных данных** (демография, экономика, инфраструктура, экология, цифровая среда, социальные услуги).  \n3. **Анализ заинтересованных сторон** (власть, бизнес, НКО, жители).  \n4. **SWOT‑анализ** города.  \n5. **Определение стратегических направлений**.  \n6. **Формулирование SMART‑целей** для каждого направления.  \n7. **Разработка мероприятий и проектов** (сроки, ответственные, ресурсы).  \n8. **Определение KPI и методики мониторинга**.  \n9. **Подготовка и публичное обсуждение стратегического документа**.  \n10. **Утверждение и план реализации** (календарный график, бюджет, распределение ответственности).  \n\n---\n\n## 2.\u202fРасширенный перечень исходных данных  \n\n| Катег

In [ ]:
print(messages[-1])

**Город Гатчина (Ленинградская область)  
Стратегический план «Целеполагание 2026‑2035»**  

---

## 1. Исходные данные и базовый профиль

### 1.1 Демографический базис (2023 г.)  

| Показатель | Значение | Источник | Период обновления |
|------------|----------|----------|-------------------|
| Население, чел. | 93 800 | Росстат, муниципальная статистика | ежегодно |
| Доля населения ≥ 65 лет, % | 18,2 % | Росстат | ежегодно |
| Естественный прирост, % | –0,4 % | Росстат | ежегодно |
| Миграционный прирост, % | +0,7 % | Росстат | ежегодно |
| Средняя продолжительность жизни, лет | 78,5 | Росстат | ежегодно |

*Обновление: каждый год, публикация в открытом муниципальном портале.*

### 1.2 Экономический базис (2023 г.)  

| Показатель | Значение | Источник | Период |
|------------|----------|----------|--------|
| ВРП муниципального уровня, млрд руб. | 12,4 | Росстат, региональная служба эконом. развития | 2010‑2025 |
| Отраслевая структура (% от ВРП) | Промышленность 30, Строительство